# Tools with MCP ⏰

The Model Context Protocol (MCP) provides a standardized way to connect AI agents to external tools and data sources. Let's connect to an MCP server using `langchain-mcp-adapters`.

## Setup

Load and/or check for needed environmental variables

In [ ]:
import os
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

In [ ]:
import sys

# MCP's Windows subprocess fallback needs a real stderr file handle.
# Jupyter replaces sys.stderr with an object whose fileno() is unsupported.
if sys.platform == "win32":
    sys.stderr = sys.__stderr__

from langchain_mcp_adapters.client import MultiServerMCPClient
import shutil
import tempfile
from pathlib import Path

# Use the official Python MCP time server. The npm server previously used here
# does not install on Windows. Resolving uvx gives subprocess an absolute path,
# which also avoids the Windows uvx.exe/command lookup difference.
uvx = shutil.which("uvx")
if uvx is None:
    raise RuntimeError("uvx was not found. Install uv from https://docs.astral.sh/uv/.")

mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": uvx,
            "args": [
                "--cache-dir",
                str(Path(tempfile.gettempdir()) / "mcp-uv-cache"),
                # mcp-server-time (mcp>=1.23.0, no upper bound) breaks against
                # mcp 2.0.0+, which renamed McpError to MCPError:
                # https://github.com/modelcontextprotocol/servers/issues/4570
                "--with",
                "mcp<2",
                "mcp-server-time",
            ],
        }
    },
)

# Load tools from the MCP server
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

Create an agent with the MCP-provided time tools.

In [ ]:
from langchain.chat_models import init_chat_model

# initialize a commercial chat model (e.g., "google_genai:gemini-2.5-flash") or use a local
# model (e.g., "ollama:gpt-oss")
model = init_chat_model(os.getenv("AI_MODEL", "ollama:gpt-oss"), temperature=0)

In [ ]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model=model,
    tools=mcp_tools,
    system_prompt="You are a helpful assistant",
)

Ask about the current time in San Francisco.

In [ ]:
result = await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "What's the time in SF right now?"}]}
)
for msg in result["messages"]:
    msg.pretty_print()